# 03 — ASR: Wav2Vec2 Fine-tuning (Kaggle / Colab / Local)

Fine-tunes `nguyenvulebinh/wav2vec2-base-vi-vlsp2020` trên ViMD.

## ✅ Fixes applied

| # | Bug | Fix |
|---|-----|-----|
| 1 | AMP + CTC → NaN/inf | `USE_AMP=False` |
| 2 | LR quá thấp → model kẹt blank mode | `LR=3e-5` (tăng từ 1e-5) |
| 3 | GRAD_ACC quá cao → warmup kéo dài | `GRAD_ACC=4` (giảm từ 8) |
| 4 | `scheduler.step()` sai vị trí | Step sau mỗi `optimizer.step()` |
| 5 | audio trim → sample rỗng | zeros fallback |
| 6 | Label không giới hạn độ dài | `MAX_LABEL_LEN=256` |
| 7 | Không chặn NaN trước backward | Skip batch nếu NaN/inf |
| 8 | `freeze_feature_extractor()` deprecated | `requires_grad_(False)` |
| 9 | Thiếu `attention_mask` | `return_attention_mask=True` |
| 10 | `skip_special_tokens` thiếu | Fixed |
| 11 | Blank pred: thiếu logit diagnostic | Thêm top-k token debug |
| 12 | Colab sys.path không có `_env` | Thêm multi-env path detection |

**Config**: 3 epochs test · Colab T4 · `USE_AMP=False` · `LR=3e-5`

In [ ]:
# ── Cell 0: Environment + Paths ───────────────────────────────────────────────
import sys, os

# ── Thêm đường dẫn vào sys.path (Kaggle / Colab / Local) ─────────────────────
def _add_to_path(p):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# Kaggle
_add_to_path('/kaggle/working/project/training/notebooks')
_add_to_path('/kaggle/working/project')
# Colab (sau khi clone repo về /content/project)
_add_to_path('/content/project/training/notebooks')
_add_to_path('/content/project')
# Colab fallback (code trong Drive)
_add_to_path('/content/drive/MyDrive/GenAI_ViMD/training/notebooks')
# Local
_add_to_path(os.path.abspath('.'))

from _env import *
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ENV          : {ENV_NAME}')
print(f'Device       : {device}')
if device == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  Không có GPU! → Colab: Runtime → Change runtime type → T4 GPU')
print(f'DATASET_PATH : {FINAL_DATASET}')
print(f'AUDIO_DIR    : {AUDIO_DIR}')
print(f'CHECKPOINT   : {ASR_CKPT_DIR}')

if IS_COLAB:
    print('\nℹ️  Colab: Đảm bảo đã chạy Cell 6 "Audio Cache" trong 00_setup_colab')
    print(f'   Audio local: {AUDIO_DIR} — exists: {os.path.isdir(AUDIO_DIR)}')

In [ ]:
# ── Cell 1: Dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.40.0', 'datasets>=2.18.0', 'accelerate>=0.28.0',
                'evaluate>=0.4.0', 'jiwer>=3.0.0', 'librosa', 'soundfile'], check=False)
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Load Dataset ──────────────────────────────────────────────────────
from datasets import load_from_disk

if not os.path.exists(FINAL_DATASET):
    raise RuntimeError(
        f'Dataset không tìm thấy: {FINAL_DATASET}\n'
        f'→ Chạy 01_dataset_prep.ipynb trước.'
    )

ds = load_from_disk(FINAL_DATASET)
print('Dataset:', ds)

n_audio = len(os.listdir(AUDIO_DIR)) if os.path.isdir(AUDIO_DIR) else 0
print(f'AUDIO_DIR: {AUDIO_DIR} → {n_audio:,} files')
if n_audio == 0:
    if IS_COLAB:
        print('⚠️  /content/audio_cache/ rỗng!')
        print('   → Chạy Cell 6 "Audio Cache" trong 00_setup_colab.ipynb')
    else:
        print('⚠️  Audio dir rỗng — chạy lại 01_dataset_prep Cell 5')

In [ ]:
# ── Cell 3: Load Model + Processor ────────────────────────────────────────────
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vi-vlsp2020'
processor  = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model      = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
).to(device)

# ✅ Fix 8: freeze_feature_extractor() deprecated >= 4.37
try:
    model.freeze_feature_extractor()
    print('freeze_feature_extractor() OK')
except AttributeError:
    model.wav2vec2.feature_extractor.requires_grad_(False)
    print('✓ Feature extractor frozen (requires_grad_(False))')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}%)')
print(f'Vocab size : {len(processor.tokenizer)}')
print(f'Blank id   : {processor.tokenizer.pad_token_id}')

In [ ]:
# ── Cell 4: Tokenizer Diagnostic ──────────────────────────────────────────────
# ✅ Fix 10: kiểm tra UNK trước khi train
TEXT_COL = 'text' if 'text' in ds['train'].column_names else 'transcript'
print(f'TEXT_COL: {TEXT_COL}')
print(f'Vocab: {len(processor.tokenizer)} | pad_id: {processor.tokenizer.pad_token_id}')

unk_found = False
for t in ds['train'][:5][TEXT_COL]:
    if not t or not str(t).strip(): continue
    t = str(t).strip().lower()
    ids = processor.tokenizer(t).input_ids
    unk = processor.tokenizer.unk_token_id
    has_unk = unk is not None and unk in ids
    if has_unk: unk_found = True
    print(f'  {t[:50]} → {"❌UNK" if has_unk else "✓"}')

if unk_found:
    print('\n❌ UNK tokens phát hiện — cần kiểm tra text normalization')
else:
    print('\n✓ Tokenizer OK')

In [ ]:
# ── Cell 5: DataLoader + Collate ──────────────────────────────────────────────
import numpy as np, librosa, soundfile as sf, io
import torch
from torch.utils.data import DataLoader

MAX_SAMPLES   = int(10.0 * 16000)  # 10s
MAX_LABEL_LEN = 256                # ✅ Fix 6: giới hạn label length

def read_audio(fn: str) -> np.ndarray:
    fpath = os.path.join(AUDIO_DIR, fn)
    with open(fpath, 'rb') as f:
        b = f.read()
    data, sr = sf.read(io.BytesIO(b), dtype='float32')
    if data.ndim > 1: data = data.mean(axis=1)
    if sr != 16000: data = librosa.resample(data, orig_sr=sr, target_sr=16000)
    # ✅ Fix 5: audio trim rỗng → zeros
    data, _ = librosa.effects.trim(data, top_db=20)
    if len(data) == 0:
        data = np.zeros(1600, dtype=np.float32)
    return data[:MAX_SAMPLES].astype(np.float32)

def collate_fn(batch):
    audios, labels = [], []
    for item in batch:
        try: audios.append(read_audio(item['filename']))
        except Exception: audios.append(np.zeros(1600, dtype=np.float32))
        t   = str(item.get(TEXT_COL, item.get('text', ''))).strip().lower()
        ids = processor.tokenizer(t).input_ids[:MAX_LABEL_LEN] if t else []
        labels.append(ids if ids else [processor.tokenizer.pad_token_id])

    # ✅ Fix 9: return_attention_mask=True
    inputs = processor(audios, sampling_rate=16000, return_tensors='pt',
                       padding=True, return_attention_mask=True)
    max_ll = max(len(l) for l in labels)
    inputs['labels'] = torch.tensor(
        [l + [-100] * (max_ll - len(l)) for l in labels], dtype=torch.long)
    return inputs

def has_audio(item):
    return os.path.exists(os.path.join(AUDIO_DIR, item['filename']))

ds_train = ds['train'].filter(has_audio, desc='Filter train')
ds_valid = ds['valid'].filter(has_audio, desc='Filter valid')
ds_test  = ds['test'].filter(has_audio,  desc='Filter test')
print(f'Train: {len(ds_train):,} | Valid: {len(ds_valid):,} | Test: {len(ds_test):,}')

if len(ds_train) == 0:
    raise RuntimeError(
        f'Không có audio file nào match trong {AUDIO_DIR}!\n'
        f'→ Kiểm tra AUDIO_DIR có đúng không (Colab: chạy Cell 6 trong 00_setup_colab)'
    )

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(ds_valid, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS)
print(f'BATCH_SIZE={BATCH_SIZE} | Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}')

# Quick verify
s = next(iter(DataLoader(ds_train.select(range(min(2,len(ds_train)))),
                          batch_size=2, collate_fn=collate_fn)))
assert 'attention_mask' in s, '❌ attention_mask MISSING!'
print(f'✓ attention_mask shape: {s["attention_mask"].shape}')
print(f'✓ input_values shape : {s["input_values"].shape}')
print(f'✓ labels shape       : {s["labels"].shape}')

In [ ]:
# ── Cell 6: Training Loop ─────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import evaluate, json

wer_metric = evaluate.load('wer')

# ─── Hyperparameters ──────────────────────────────────────────────────────────
NUM_EPOCHS    = 3            # test run — tăng lên 15 sau khi verify WER giảm
# ✅ Fix 2: LR=3e-5 (tăng từ 1e-5) — CTC cần LR đủ lớn để thoát blank mode
LR            = 3e-5
# ✅ Fix 3: GRAD_ACC=4 (giảm từ 8) — effective batch 32, warmup hợp lý hơn
GRAD_ACC      = 4
MAX_GRAD_NORM = 1.0
# Warmup ngắn hơn để model bắt đầu học sớm
WARMUP_STEPS  = 50
# ✅ Fix 1: AMP=False vì fp16 + CTC sinh inf → nan
USE_AMP       = False

# ─── Resume từ checkpoint ─────────────────────────────────────────────────────
CKPT_STATE = os.path.join(ASR_CKPT_DIR, 'train_state.json')
start_epoch = 0
best_wer    = float('inf')

if os.path.exists(CKPT_STATE):
    with open(CKPT_STATE) as f:
        ckpt = json.load(f)
    start_epoch = ckpt.get('epoch', 0)
    best_wer    = ckpt.get('best_wer', float('inf'))
    model_ckpt  = os.path.join(ASR_CKPT_DIR, 'config.json')
    if os.path.exists(model_ckpt):
        model = Wav2Vec2ForCTC.from_pretrained(ASR_CKPT_DIR).to(device)
        try: model.freeze_feature_extractor()
        except AttributeError: model.wav2vec2.feature_extractor.requires_grad_(False)
        print(f'✅ Resume: epoch={start_epoch}, best_wer={best_wer:.2f}%')
    else:
        start_epoch = 0

if start_epoch >= NUM_EPOCHS:
    print(f'✅ Training done ({start_epoch} epochs). Chạy Cell 8.')
else:
    optimizer   = AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=LR, weight_decay=0.01, betas=(0.9, 0.98))
    total_steps = NUM_EPOCHS * max(1, len(train_loader) // GRAD_ACC)
    # ✅ Fix 4: LinearScheduler với warmup
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS,
        num_training_steps=max(total_steps, 1))

    if start_epoch > 0:
        for _ in range(start_epoch * max(1, len(train_loader) // GRAD_ACC)):
            scheduler.step()

    # ✅ Fix 1: scaler chỉ dùng khi USE_AMP=True
    scaler = torch.cuda.amp.GradScaler() if (USE_AMP and device == 'cuda') else None

    eff_batch = BATCH_SIZE * GRAD_ACC
    print(f'Training config:')
    print(f'  Epochs         : {NUM_EPOCHS} (resume from {start_epoch})')
    print(f'  LR             : {LR}')
    print(f'  Eff.batch      : {BATCH_SIZE}×{GRAD_ACC}={eff_batch}')
    print(f'  Warmup steps   : {WARMUP_STEPS}')
    print(f'  Total steps    : {total_steps:,}')
    print(f'  USE_AMP        : {USE_AMP}')
    print()

    global_step = start_epoch * max(1, len(train_loader) // GRAD_ACC)
    nan_count   = 0

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        total_loss = 0.0
        valid_steps = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            iv   = batch['input_values'].to(device)
            mask = batch['attention_mask'].to(device)   # ✅ Fix 9
            lbl  = batch['labels'].to(device)

            # ✅ Fix 1: USE_AMP=False → không dùng autocast
            if scaler and USE_AMP:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(input_values=iv, attention_mask=mask, labels=lbl)
                loss = outputs.loss / GRAD_ACC
                # ✅ Fix 7: NaN check trước khi backward
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1; optimizer.zero_grad(); continue
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACC == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    # ✅ Fix 4: scheduler.step() sau optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1
            else:
                outputs = model(input_values=iv, attention_mask=mask, labels=lbl)
                loss    = outputs.loss / GRAD_ACC
                # ✅ Fix 7: NaN check
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1; optimizer.zero_grad(); continue
                loss.backward()
                if (step + 1) % GRAD_ACC == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    optimizer.step()
                    # ✅ Fix 4: step sau optimizer
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

            total_loss  += loss.item() * GRAD_ACC
            valid_steps += 1
            pbar.set_postfix(loss=f'{loss.item()*GRAD_ACC:.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}', nan=nan_count)

        avg_loss = total_loss / valid_steps if valid_steps > 0 else float('nan')

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        all_preds, all_refs = [], []
        blank_total = pred_total = 0
        first_debug = True

        with torch.no_grad():
            for vb in tqdm(val_loader, desc='Val', leave=False):
                iv_v    = vb['input_values'].to(device)
                mask_v  = vb['attention_mask'].to(device)
                logits  = model(input_values=iv_v, attention_mask=mask_v).logits
                pred_ids = logits.argmax(dim=-1)

                # ✅ Fix 11: group_tokens=True (default) để CTC collapse blank
                preds   = processor.batch_decode(pred_ids, group_tokens=True)

                vlbl    = vb['labels'].clone()
                vlbl[vlbl == -100] = processor.tokenizer.pad_token_id
                # ✅ Fix 10: skip_special_tokens
                refs    = processor.tokenizer.batch_decode(vlbl, skip_special_tokens=True)
                all_preds.extend(preds); all_refs.extend(refs)

                blank_id    = processor.tokenizer.pad_token_id
                blank_total += (pred_ids == blank_id).sum().item()
                pred_total  += pred_ids.numel()

                # ✅ Fix 11: In diagnostic rõ ràng hơn + logit confidence
                if first_debug:
                    print(f'\n[Val Debug] Epoch {epoch+1} — sample predictions:')
                    for pi, (p, r) in enumerate(zip(preds[:3], refs[:3])):
                        print(f"  PRED: '{p[:60] if p else '(blank)'}")
                        print(f"  REF : '{r[:60]}'")
                    # Logit confidence: xem model đang predict token nào nhiều nhất
                    top_ids = pred_ids[0].cpu().tolist()
                    unique, counts = zip(*[
                        (tid, top_ids.count(tid))
                        for tid in sorted(set(top_ids), key=top_ids.count, reverse=True)[:5]
                    ])
                    print(f'  Top predicted token IDs: {list(zip(unique, counts))}')
                    print(f'  (blank_id={blank_id})')
                    first_debug = False

        blank_ratio = blank_total / pred_total if pred_total > 0 else 1.0
        pairs = [(p, r) for p, r in zip(all_preds, all_refs) if r.strip()]

        # ── Tính WER: nếu tất cả pred rỗng, thay bằng space để tránh jiwer crash
        safe_preds = [p if p.strip() else ' ' for p, r in pairs]
        safe_refs  = [r for p, r in pairs]
        wer = 100 * wer_metric.compute(
            predictions=safe_preds, references=safe_refs
        ) if pairs else 100.0

        print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}: '
              f'loss={avg_loss:.4f} | WER={wer:.2f}% | '
              f'blank={blank_ratio:.1%} | NaN={nan_count}')

        if blank_ratio > 0.95 and epoch == 0:
            print('  ℹ️  Blank>95% epoch 1 — bình thường với CTC')
            print('  ℹ️  Nếu epoch 2-3 vẫn blank>95%: kiểm tra LR và attention_mask')
        elif blank_ratio > 0.95 and epoch >= 2:
            print('  ⚠️  Blank>95% sau epoch 3 — kiểm tra:')
            print('      1. AUDIO_DIR có đúng không?')
            print('      2. attention_mask được truyền vào model không?')
            print('      3. Thử tăng LR lên 5e-5')

        # Save checkpoint
        if wer < best_wer:
            best_wer = wer
            model.save_pretrained(ASR_CKPT_DIR)
            processor.save_pretrained(ASR_CKPT_DIR)
            print(f'  ✅ Best checkpoint saved (WER={wer:.2f}%)')

        with open(CKPT_STATE, 'w') as f:
            json.dump({'epoch': epoch+1, 'best_wer': best_wer,
                       'global_step': global_step, 'nan_count': nan_count,
                       'lr': LR, 'grad_acc': GRAD_ACC}, f, indent=2)

        if IS_COLAB:
            print(f'  💾 State saved → Drive (persist sau runtime reset)')
        elif IS_KAGGLE:
            print(f'  💾 State saved → {CKPT_STATE}')

    print(f'\n✅ Training done. Best WER: {best_wer:.2f}%')
    if IS_KAGGLE:
        print('\n⚠️  Kaggle: Nhấn "Save Version" để persist checkpoint!')

In [ ]:
# ── Cell 7: Export best model ─────────────────────────────────────────────────
import shutil
if os.path.exists(os.path.join(ASR_CKPT_DIR, 'config.json')):
    os.makedirs(ASR_WEIGHT_DIR, exist_ok=True)
    for fn in os.listdir(ASR_CKPT_DIR):
        src = os.path.join(ASR_CKPT_DIR, fn)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(ASR_WEIGHT_DIR, fn))
    print(f'✅ Exported → {ASR_WEIGHT_DIR}')
else:
    print('⚠️  Chưa có checkpoint — chạy Cell 6 trước')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn "Save Version" để lưu model!')
elif IS_COLAB:
    print('\n✓ Colab: Model đã export vào Drive — persist sau runtime reset')

In [ ]:
# ── Cell 8: Test Set Evaluation ───────────────────────────────────────────────
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

eval_src = ASR_WEIGHT_DIR if os.path.exists(os.path.join(ASR_WEIGHT_DIR, 'config.json')) else ASR_CKPT_DIR
eval_model = Wav2Vec2ForCTC.from_pretrained(eval_src).to(device)
eval_proc  = Wav2Vec2Processor.from_pretrained(eval_src)
eval_model.eval()

wer2 = evaluate.load('wer')
all_preds, all_refs = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        logits   = eval_model(input_values=batch['input_values'].to(device),
                              attention_mask=batch['attention_mask'].to(device)).logits
        pred_ids = logits.argmax(dim=-1)
        preds    = eval_proc.batch_decode(pred_ids, group_tokens=True)
        lbl      = batch['labels'].clone()
        lbl[lbl == -100] = eval_proc.tokenizer.pad_token_id
        refs     = eval_proc.tokenizer.batch_decode(lbl, skip_special_tokens=True)
        all_preds.extend(preds); all_refs.extend(refs)

pairs = [(p, r) for p, r in zip(all_preds, all_refs) if r.strip()]
safe_preds = [p if p.strip() else ' ' for p, r in pairs]
safe_refs  = [r for p, r in pairs]
test_wer = 100 * wer2.compute(
    predictions=safe_preds, references=safe_refs
) if pairs else 100.0

print(f'\n===== Test WER: {test_wer:.2f}% =====')
print(f'Target: ≤ 8.0% | Status: {"✅ PASS" if test_wer <= 8.0 else "📈 In progress"}')
print('\nSample predictions:')
for p, r in list(zip(all_preds[:4], all_refs[:4])):
    print(f'  PRED: {repr(p[:60])}')
    print(f'  REF : {repr(r[:60])}')

## Kết quả kỳ vọng (với LR=3e-5, GRAD_ACC=4)

| Epoch | Blank ratio | WER (kỳ vọng) |
|-------|------------|---------------|
| 1 | ~90-100% | ~100% (CTC học blank) |
| 2 | ~50-80% | 60-100% |
| 3 | <50% | 30-70% |
| 10+ | <10% | < 20% |

**Nếu WER epoch 3 bắt đầu giảm** → tăng `NUM_EPOCHS = 15` và re-run Cell 6.

**Nếu blank vẫn 100% sau epoch 2** → thử `LR=5e-5` hoặc kiểm tra AUDIO_DIR.

**Colab**: Checkpoint lưu vào Drive → persist sau runtime reset.